In [ ]:
import os
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.classification import DecisionTreeClassifier

In [ ]:
spark = SparkSession.builder.appName('classification').getOrCreate()

In [ ]:
display(os.listdir('work'))

## Create Dataframe

In [ ]:
data = spark.read.csv('work/hotel_reservations.csv', inferSchema=True, header=True)
data.count()

In [ ]:
(
   data.select
   (
       [
           F.count(
               F.when(
                   F.col(c).isNull(),1
               )
           ).alias(c) for c in data.columns
       ]
   ).show()
)   

In [ ]:
data.show()

In [ ]:
training_data, testing_data = data.randomSplit([0.8,0.2], seed=42)

## Create Features

In [ ]:
pipeline_stages = []

In [ ]:
categorical_columns = ['type_of_meal_plan','room_type_reserved','arrival_month','market_segment_type']

for col in categorical_columns:
    string_indexer = StringIndexer(inputCol=col, outputCol=col + '_index')
    print(f'StringIndexer {string_indexer.getInputCol()} -> {string_indexer.getOutputCol()}')

    encoder = OneHotEncoder(inputCol=string_indexer.getOutputCol(), outputCol=col + '_vec', dropLast=False)
    print(f'OneHotEncoder {encoder.getInputCol()} -> {encoder.getOutputCol()}')
    print()
    pipeline_stages += [string_indexer, encoder]

In [ ]:
pipeline_stages

In [ ]:
pipeline_stages += [StringIndexer(inputCol='booking_status', outputCol='booking_status_index')]

In [ ]:
encoded_categorical_cols = [col + '_vec' for col in categorical_columns]
encoded_categorical_cols

In [ ]:
numerical_columns = [
  'no_of_adults',
  'no_of_children',
  'no_of_weekend_nights',
  'no_of_week_nights',
  'required_car_parking_space',
  'lead_time',
  'repeated_guest',
  'no_of_previous_cancellations',
  'no_of_previous_bookings_not_canceled',
  'avg_price_per_room',
  'no_of_special_requests'
]

In [ ]:
input_columns = encoded_categorical_cols + numerical_columns
input_columns

In [ ]:
assembler = VectorAssembler(inputCols=input_columns, outputCol='features')
pipeline_stages.append(assembler)

## Create Algorithm Object

In [ ]:
dtc = DecisionTreeClassifier(featuresCol='features', labelCol='booking_status_index')
pipeline_stages.append(dtc)

In [ ]:
pipeline_stages

## Create  Pipeline and Prediction

In [ ]:
pipeline = Pipeline(stages=pipeline_stages)
model = pipeline.fit(training_data)
prediction = model.transform(testing_data)
prediction.select('features','booking_status_index','prediction').show(20)

## Create Evaluations

In [ ]:
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol='booking_status_index', predictionCol='prediction', metricName='accuracy')
accuracy = accuracy_evaluator.evaluate(prediction) * 100
print(f'Accuracy: {accuracy:.2f}%')

## Create Hyperparameters

In [ ]:
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder

In [ ]:
paramGrid = (
    ParamGridBuilder()
    .addGrid(dtc.maxDepth, [2,5,10])
    .addGrid(dtc.maxBins, [10,20,40])
    .addGrid(dtc.minInstancesPerNode, [1,2,4])
    .build()
)

evaluator = MulticlassClassificationEvaluator(
    labelCol='booking_status_index',
    predictionCol='prediction',
    metricName='accuracy'
)

tvs_model = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    seed=42
)

best_model = tvs_model.fit(training_data)
prediction = best_model.transform(testing_data)
accuracy = evaluator.evaluate(prediction) * 100
print(f'Best Model Accuracy: {accuracy:.2f}%')